# 04 · asyncio: cooperating while waiting

**Time:** 25–30 minutes, including the exercises.
**Before you start:** Know functions, loops, and the difference between waiting and CPU work.

You will use coroutines, schedule tasks, avoid blocking the event loop, limit active operations,
and stop a task cleanly. All examples run offline with Python's standard library.

Run the cells in order in a Python 3.10+ Jupyter kernel. Jupyter already runs an event loop,
so these cells use **top-level `await`**: `await` directly in a cell, outside a
function. Do not wrap them in `asyncio.run()`.
In a regular Python script, you would normally put the async work in `async def main()` and
call `asyncio.run(main())` once when starting the program.

[Course guide](README.md) · Previous: [Processes](03_processes_and_cpu_work.ipynb) ·
Next: [Backpressure](05_backpressure_and_bounded_queues.ipynb)

[IPython's notebook async support](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html#using-autoawait-in-a-notebook-ipykernel)

In [1]:
import asyncio
import time

## 1. Three terms, one small example

- A **coroutine function** is defined with `async def`. Calling it creates a
  **coroutine object**: async work that can be run by awaiting or scheduling it.
- A **task** schedules a coroutine to run on the event loop.
- The **event loop** runs ready tasks and resumes them when their waits finish.

`await` waits for an async result. When that operation is still pending, the task can pause and
let other tasks use the loop. This is **cooperative scheduling**: each task
returns control to the loop so other work can progress. An `await` on an
already-ready result need not pause.
[Coroutines and tasks](https://docs.python.org/3/library/asyncio-task.html#coroutines)

**Predict:** Does calling `pretend_fetch()` below immediately run its body?

In [2]:
async def pretend_fetch(name, delay=0.12):
    # Simulate an async I/O wait; no network request happens.
    await asyncio.sleep(delay)
    return f"data for {name}"

coroutine = pretend_fetch("sample_0")
print("Calling async def created:", type(coroutine).__name__)
result = await coroutine
print("After awaiting:", result)

Calling async def created: coroutine
After awaiting: data for sample_0


**Explain:** The function call created work we could await. It did not run the body at the call itself.
We await this coroutine exactly once. Call the function again to make another coroutine for another run.

`asyncio.sleep()` stands in for an async operation that has to wait.
This lesson teaches coordination, not actual network performance.

## 2. Sequential awaits still run one job at a time

**Predict:** Four calls each wait for 0.12 seconds. Will this loop overlap them?

In [3]:
names = [f"sample_{index}" for index in range(4)]
started = time.perf_counter()
sequential_results = []
for name in names:
    sequential_results.append(await pretend_fetch(name))
sequential_seconds = time.perf_counter() - started

print(sequential_results)
print(f"Sequential awaits: {sequential_seconds:.3f} seconds")

['data for sample_0', 'data for sample_1', 'data for sample_2', 'data for sample_3']
Sequential awaits: 0.526 seconds


**Explain:** Each fetch finishes before the next is created. The waiting adds up to about 0.48 seconds.
Other tasks elsewhere on the loop could still run, but these four fetches do not overlap each other.

## 3. Schedule several jobs with gather

`gather()` schedules the supplied coroutines and collects their results in input order.
The `*` below passes each coroutine as a separate argument.
[Gather documentation](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather)

**Predict:** About how much waiting will these four jobs need when their waits overlap?

In [4]:
started = time.perf_counter()
concurrent_results = await asyncio.gather(
    *(pretend_fetch(name) for name in names)
)
concurrent_seconds = time.perf_counter() - started

print(concurrent_results)
print(f"Concurrent waits: {concurrent_seconds:.3f} seconds")
assert concurrent_results == sequential_results

['data for sample_0', 'data for sample_1', 'data for sample_2', 'data for sample_3']
Concurrent waits: 0.128 seconds


**Explain:** Expect closer to 0.12 seconds plus overhead. Actual timing varies.
There are several pending jobs, but this example uses one event-loop thread.

We can also schedule a task explicitly with `create_task()`. Keep a reference and await it
so its result or failure is handled. Here, the fetch progresses while the caller does other async work.

In [5]:
task = asyncio.create_task(pretend_fetch("scheduled_sample"))
await asyncio.sleep(0.03)  # The caller is also waiting, so the loop can run the fetch.
print("The caller can do something else before collecting the result.")
print(await task)

The caller can do something else before collecting the result.


data for scheduled_sample


## 4. Blocking the loop blocks other tasks

`time.sleep()` blocks the thread that calls it. `asyncio.sleep()` pauses the current task
and lets the event loop run other tasks.
Writing `async def` around blocking code does not make that code nonblocking.

We will run a **heartbeat** alongside a short operation. A heartbeat prints at intervals so
we can notice when the loop gets stuck. Both demonstrations end on their own.

**Predict:** Which run will have a long gap between the first two heartbeats?

In [6]:
async def heartbeat(started):
    for beat in range(4):
        elapsed = time.perf_counter() - started
        print(f"  heartbeat {beat} at {elapsed:.2f}s")
        await asyncio.sleep(0.04)

async def blocking_wait():
    time.sleep(0.20)  # Deliberately wrong inside the event-loop thread.

async def cooperative_wait():
    await asyncio.sleep(0.20)

async def demonstrate_wait(label, operation):
    print(label)
    started = time.perf_counter()
    pulse = asyncio.create_task(heartbeat(started))
    await asyncio.sleep(0)  # Give the heartbeat a turn before the operation.
    await operation()
    await pulse  # Leave no heartbeat running after this demo.

await demonstrate_wait("Blocking sleep:", blocking_wait)
await demonstrate_wait("Async sleep:", cooperative_wait)

Blocking sleep:
  heartbeat 0 at 0.00s


  heartbeat 1 at 0.20s


  heartbeat 2 at 0.24s
  heartbeat 3 at 0.29s
Async sleep:
  heartbeat 0 at 0.00s
  heartbeat 1 at 0.05s


  heartbeat 2 at 0.10s


  heartbeat 3 at 0.15s


**Explain:** During `time.sleep(0.20)`, the loop's thread cannot run the heartbeat.
During `await asyncio.sleep(0.20)`, it can. The printed intervals are approximate, not promises.

A long pure Python CPU loop would also block the event loop. Use processes for CPU-heavy Python
work when appropriate; asyncio itself does not spread that work across CPU cores.

## 5. Use a thread for a blocking I/O function

Suppose a library only provides a normal blocking function. `asyncio.to_thread()` lets us await
that function while it runs in a worker thread. The event-loop thread remains available.
[To-thread documentation](https://docs.python.org/3/library/asyncio-task.html#asyncio.to_thread)

In [7]:
def blocking_read():
    time.sleep(0.20)  # Simulated blocking file or network wait.
    return "data from a blocking function"

async def read_using_thread():
    data = await asyncio.to_thread(blocking_read)
    print(" ", data)

await demonstrate_wait("Blocking function moved to a thread:", read_using_thread)

Blocking function moved to a thread:
  heartbeat 0 at 0.00s
  heartbeat 1 at 0.05s
  heartbeat 2 at 0.10s


  heartbeat 3 at 0.15s


  data from a blocking function


**Explain:** The heartbeat continues because the wait occurs on another thread.
On a normal CPython build with the GIL enabled, this does not turn a pure Python CPU loop
into parallel CPU work. Cancelling an await of `to_thread()` also does not forcibly stop a
function that is already running in its thread.

## 6. Limit active operations with a semaphore

A **semaphore** is a limited set of permits. With two permits, at most two tasks can be inside
its protected block. A task that needs a permit waits without blocking the loop.
[Semaphore documentation](https://docs.python.org/3/library/asyncio-sync.html#asyncio.Semaphore)

The limit must be a positive integer. Zero permits would leave every fetch
waiting forever; our helper rejects that value. `type(limit) is int` checks
that the supplied value is an integer.

**Predict:** We schedule six tasks below. How many can be fetching at once?

In [8]:
async def fetch_with_limit(items, limit):
    if type(limit) is not int or limit < 1:
        raise ValueError("limit must be a positive integer")
    semaphore = asyncio.Semaphore(limit)
    active = 0
    peak_active = 0

    async def limited_fetch(name):
        nonlocal active, peak_active
        async with semaphore:
            active += 1
            peak_active = max(peak_active, active)
            try:
                return await pretend_fetch(name, delay=0.06)
            finally:
                active -= 1

    results = await asyncio.gather(*(limited_fetch(name) for name in items))
    return results, peak_active

limited_results, peak = await fetch_with_limit(range(6), limit=2)
print("Results:", limited_results)
print("Largest number active:", peak)
assert len(limited_results) == 6
assert peak <= 2

Results: ['data for 0', 'data for 1', 'data for 2', 'data for 3', 'data for 4', 'data for 5']
Largest number active: 2


**Explain:** Only two fetches can hold permits at once. `async with` returns a permit when the block exits.
`finally` runs when the block exits, including on an exception or cancellation,
so our counter is updated in each case.
`nonlocal` means the nested function updates the variables in `fetch_with_limit`.

This example still creates **six tasks**. A semaphore limits active operations; it does not limit
the number of waiting task objects or the size of the collected results. Creating a million tasks
up front can consume substantial memory even with a semaphore.

For a large stream, use a bounded queue and a fixed number of consumers, or process a bounded group
at a time. That is part of **backpressure**, the topic of the next lesson.

## 7. Cancel a task and let it clean up

Cancellation requests that a task stop. The task receives `CancelledError`, usually while it is paused at an `await`.
Use `finally` for cleanup and await the cancelled task to finish handling it.
[Task cancellation](https://docs.python.org/3/library/asyncio-task.html#task-cancellation)

Our `Event` is a small signal: the caller waits until the worker has started before cancelling it.
It keeps this demonstration independent of timing guesses.

In [9]:
async def cancellation_demo():
    started = asyncio.Event()

    async def worker():
        try:
            started.set()
            await asyncio.sleep(0.50)
            return "finished normally"
        finally:
            print("Worker cleanup ran")

    task = asyncio.create_task(worker())
    await started.wait()
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("Caller confirmed cancellation")

await cancellation_demo()

Worker cleanup ran
Caller confirmed cancellation


**Explain:** The worker cleans up, then lets cancellation reach the caller.
Cancellation is cooperative: it cannot interrupt arbitrary blocking Python code immediately.

Exceptions also come back through `await`. With default `gather()`, the first worker exception is
raised to the caller, but other tasks are not automatically cancelled. For small, independent jobs,
`gather(..., return_exceptions=True)` is one way to wait for all outcomes and inspect each one.
On Python 3.11+, `TaskGroup` can cancel the other tasks in its group after a
failure. [Lesson 09](09_failures_and_clean_shutdown.ipynb) explains that pattern.

## Exercises

1. Fetch six items with semaphore limits of one, two, and three. Predict how many waves each needs.
   Check the returned item count and the largest number active. No exact timing assertion is needed.
2. A coroutine calls a blocking library function named `load_metadata(path)`. Rewrite that call so
   the event loop stays available, assuming the function mainly waits for I/O.
3. A colleague uses a semaphore of four but creates a task for each of ten million images.
   Explain what is still unbounded and suggest a better shape for the program.
4. Why does an `await` inside a loop not necessarily make that loop's jobs overlap?

In [10]:
# Your work. Reuse fetch_with_limit and asyncio.to_thread.
exercise_items = [f"image_{index}" for index in range(6)]

<details>
<summary>Worked answers — open after trying</summary>

**1.** Expect six, three, and two waves. Scheduling overhead can affect elapsed time.

```python
for limit in (1, 2, 3):
    results, peak = await fetch_with_limit(exercise_items, limit)
    print("limit:", limit, "results:", len(results), "peak:", peak)
    assert len(results) == 6
    assert peak <= limit
```

**2.** Inside the coroutine:

```python
metadata = await asyncio.to_thread(load_metadata, path)
```

Pass the function and its arguments. Writing `load_metadata(path)` before passing it would
run the blocking call immediately in the event-loop thread.

**3.** All ten million task objects can exist at once even though only four
fetches are active. Those waiting tasks and retained results can fill memory. Use a bounded input queue, a fixed set of consumers, and a destination
(often called a **sink**) that saves or uses results as they arrive. A full queue makes the producer wait.

**4.** `await` waits for that result before continuing the loop. Overlap requires other work
to be scheduled, for example through `gather()` or `create_task()`.

</details>

## What to remember

- Async code cooperates by handing control back while waiting. It is useful for many I/O operations.
- Schedule several jobs to overlap their waits. `async def` alone does not create concurrency.
- Blocking calls stall the loop. Use async libraries or move blocking I/O to a thread.
- Limit both active work and queued work. A semaphore solves only the first problem.
- Await the tasks you create and allow cleanup when they are cancelled.

**Machine learning engineer (MLE) connection:** Async code can coordinate
requests for **metadata**, such as image dimensions and labels, or calls to a
remote model service.
It does not make image decoding or model training automatically parallel.

Next: [05 · Backpressure and bounded queues](05_backpressure_and_bounded_queues.ipynb).

**Official references:** [Coroutines and tasks](https://docs.python.org/3/library/asyncio-task.html),
[Async synchronization](https://docs.python.org/3/library/asyncio-sync.html),
[Jupyter/IPython async support](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html).